# IoT Anomaly Detection — PySpark Lakehouse Pipeline

A production-grade batch pipeline that mirrors a Databricks job, running entirely locally:

1. **Bronze** — read raw sensor JSON from the MinIO `sensor-data-lake` bucket over `s3a://`
2. **Quality gate** — quarantine malformed / out-of-range records
3. **Silver** — rolling per-machine statistics (trailing window mean & stddev)
4. **Gold** — flag readings deviating **> 3σ** from the rolling mean, write Parquet back to MinIO

> Make sure the stack is up (`docker compose up -d --build`) and the generator has been
> running for a couple of minutes so there is data to process.

In [ ]:
import os

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType
)
from pyspark.sql.window import Window

# --- Configuration (injected by docker-compose) ------------------------------
MINIO_ENDPOINT   = os.environ.get("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "minioadmin123")
BUCKET           = os.environ.get("BUCKET_NAME", "sensor-data-lake")

RAW_PATH        = f"s3a://{BUCKET}/raw/"
PROCESSED_PATH  = f"s3a://{BUCKET}/processed/readings/"
ANOMALIES_PATH  = f"s3a://{BUCKET}/processed/anomalies/"
QUARANTINE_PATH = f"s3a://{BUCKET}/quarantine/"

ROLLING_WINDOW_SIZE = 20   # trailing readings per machine
SIGMA_THRESHOLD     = 3.0  # |x - mean| > 3σ  =>  anomaly
MIN_WINDOW_SAMPLES  = 10   # don't flag until the window has enough history

In [ ]:
# --- Spark session wired to MinIO via the S3A connector -----------------------
# hadoop-aws 3.3.4 matches the Hadoop build shipped with Spark 3.5.0.
spark = (
    SparkSession.builder
    .appName("iot-anomaly-detection")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} connected to {MINIO_ENDPOINT}")

## 1 · Bronze — ingest raw JSON

An **explicit schema** is non-negotiable in production: it prevents costly schema
inference over the whole bucket and lets PERMISSIVE mode shunt anything that doesn't
conform into `_corrupt_record` instead of silently nulling fields.

In [ ]:
sensor_schema = StructType([
    StructField("reading_id",       StringType(),  True),
    StructField("machine_id",       StringType(),  True),
    StructField("timestamp",        StringType(),  True),
    StructField("temperature_c",    DoubleType(),  True),
    StructField("vibration_mm_s",   DoubleType(),  True),
    StructField("injected_anomaly", BooleanType(), True),  # ground truth, eval only
    StructField("_corrupt_record",  StringType(),  True),
])

raw_df = (
    spark.read
    .schema(sensor_schema)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(RAW_PATH)
)

# Cache before touching _corrupt_record (required by Spark's JSON reader semantics)
raw_df.cache()
total = raw_df.count()
print(f"Ingested {total:,} raw records from {RAW_PATH}")
raw_df.show(5, truncate=False)

## 2 · Quality gate — filter malformed data, quarantine the rejects

Rejected records aren't dropped on the floor — they're written to a `quarantine/`
prefix so data-quality issues stay auditable (the lakehouse equivalent of a
dead-letter queue).

In [ ]:
parsed_df = raw_df.withColumn("event_time", F.to_timestamp("timestamp"))

valid_condition = (
    F.col("_corrupt_record").isNull()
    & F.col("machine_id").isNotNull()
    & F.col("event_time").isNotNull()
    & F.col("temperature_c").isNotNull()
    & F.col("vibration_mm_s").isNotNull()
    # physically plausible ranges — beyond these it's a sensor fault, not a reading
    & F.col("temperature_c").between(-40.0, 250.0)
    & F.col("vibration_mm_s").between(0.0, 100.0)
)

clean_df    = parsed_df.filter(valid_condition).drop("_corrupt_record", "timestamp")
rejected_df = parsed_df.filter(~valid_condition | F.col("_corrupt_record").isNotNull())

n_clean, n_rejected = clean_df.count(), rejected_df.count()
print(f"Clean: {n_clean:,}  |  Quarantined: {n_rejected:,} "
      f"({100 * n_rejected / max(total, 1):.2f}% reject rate)")

(rejected_df.write.mode("overwrite").json(QUARANTINE_PATH))
print(f"Quarantine written to {QUARANTINE_PATH}")

## 3 · Silver — rolling statistics per machine

A **trailing window that excludes the current row** (`rowsBetween(-N, -1)`) gives each
reading a baseline built only from its own past — the current value can't contaminate
the statistics it's judged against.

In [ ]:
w = (
    Window.partitionBy("machine_id")
    .orderBy("event_time")
    .rowsBetween(-ROLLING_WINDOW_SIZE, -1)
)

stats_df = (
    clean_df
    .withColumn("temp_roll_mean",   F.avg("temperature_c").over(w))
    .withColumn("temp_roll_std",    F.stddev("temperature_c").over(w))
    .withColumn("vib_roll_mean",    F.avg("vibration_mm_s").over(w))
    .withColumn("vib_roll_std",     F.stddev("vibration_mm_s").over(w))
    .withColumn("window_samples",   F.count("temperature_c").over(w))
)

stats_df.select(
    "machine_id", "event_time", "temperature_c",
    "temp_roll_mean", "temp_roll_std", "window_samples"
).show(5)

## 4 · Gold — 3σ anomaly flags

A reading is anomalous when it deviates from its machine's rolling mean by more than
3 standard deviations — on either metric. Flags are suppressed until the window holds
enough history, and guarded against zero/near-zero σ.

In [ ]:
EPS = 1e-6  # guard against σ ≈ 0 on flat signals

def sigma_flag(value_col, mean_col, std_col):
    return (
        (F.col("window_samples") >= MIN_WINDOW_SAMPLES)
        & F.col(std_col).isNotNull()
        & (F.col(std_col) > EPS)
        & (F.abs(F.col(value_col) - F.col(mean_col)) > SIGMA_THRESHOLD * F.col(std_col))
    )

scored_df = (
    stats_df
    .withColumn("temp_anomaly", sigma_flag("temperature_c", "temp_roll_mean", "temp_roll_std"))
    .withColumn("vib_anomaly",  sigma_flag("vibration_mm_s", "vib_roll_mean", "vib_roll_std"))
    .withColumn("is_anomaly",   F.col("temp_anomaly") | F.col("vib_anomaly"))
    .withColumn("temp_z_score",
                F.when(F.col("temp_roll_std") > EPS,
                       (F.col("temperature_c") - F.col("temp_roll_mean")) / F.col("temp_roll_std")))
    .withColumn("vib_z_score",
                F.when(F.col("vib_roll_std") > EPS,
                       (F.col("vibration_mm_s") - F.col("vib_roll_mean")) / F.col("vib_roll_std")))
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("ingest_date",  F.to_date("event_time"))
)

anomalies_df = scored_df.filter("is_anomaly")
print(f"Anomalies flagged: {anomalies_df.count():,} of {n_clean:,} clean readings")
anomalies_df.select(
    "machine_id", "event_time", "temperature_c", "temp_z_score",
    "vibration_mm_s", "vib_z_score", "injected_anomaly"
).orderBy(F.desc("event_time")).show(10)

In [ ]:
# --- Write polished results back to the lake (Parquet, date-partitioned) ------
(scored_df.write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(PROCESSED_PATH))

(anomalies_df.write
    .mode("overwrite")
    .partitionBy("ingest_date")
    .parquet(ANOMALIES_PATH))

print(f"Scored readings -> {PROCESSED_PATH}")
print(f"Anomalies only  -> {ANOMALIES_PATH}")

## 5 · Fleet health summary

In [ ]:
summary_df = (
    scored_df.groupBy("machine_id")
    .agg(
        F.count("*").alias("readings"),
        F.round(F.avg("temperature_c"), 2).alias("avg_temp_c"),
        F.round(F.avg("vibration_mm_s"), 3).alias("avg_vib_mm_s"),
        F.sum(F.col("is_anomaly").cast("int")).alias("anomalies"),
    )
    .withColumn("anomaly_rate_pct", F.round(100 * F.col("anomalies") / F.col("readings"), 2))
    .orderBy(F.desc("anomaly_rate_pct"))
)
summary_df.show(20)

# Detection quality vs the generator's hidden ground-truth labels
(scored_df
    .groupBy("injected_anomaly", "is_anomaly")
    .count()
    .orderBy("injected_anomaly", "is_anomaly")
    .show())

## 6 · Visual check — one machine's temperature with anomalies highlighted

In [ ]:
import matplotlib.pyplot as plt

machine = scored_df.select("machine_id").first()["machine_id"]
pdf = (
    scored_df.filter(F.col("machine_id") == machine)
    .select("event_time", "temperature_c", "temp_roll_mean", "temp_roll_std", "is_anomaly")
    .orderBy("event_time")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(pdf.event_time, pdf.temperature_c, lw=0.9, color="#4a7aa7", label="temperature")
ax.plot(pdf.event_time, pdf.temp_roll_mean, lw=1.1, color="#444", ls="--", label="rolling mean")
band = 3 * pdf.temp_roll_std
ax.fill_between(pdf.event_time, pdf.temp_roll_mean - band, pdf.temp_roll_mean + band,
                alpha=0.15, color="#888", label="±3σ band")
flagged = pdf[pdf.is_anomaly]
ax.scatter(flagged.event_time, flagged.temperature_c, color="#c0392b", zorder=5,
           s=38, label="flagged anomaly")
ax.set_title(f"{machine} — temperature vs rolling 3σ envelope")
ax.set_ylabel("°C")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 7 · Optional — run the same pipeline as a Structured Stream

The batch job above is rerunnable and idempotent. For continuous operation, the same
logic runs as a Spark Structured Streaming job. `availableNow` processes everything
new since the last checkpoint and exits — the lakehouse equivalent of a Databricks
triggered job; swap it for `processingTime="30 seconds"` to run forever.

In [ ]:
RUN_STREAMING = False  # flip to True to try it

if RUN_STREAMING:
    stream_df = (
        spark.readStream
        .schema(sensor_schema)
        .option("maxFilesPerTrigger", 50)
        .json(RAW_PATH)
        .withColumn("event_time", F.to_timestamp("timestamp"))
        .filter(valid_condition)
        .drop("_corrupt_record", "timestamp")
        .withColumn("ingest_date", F.to_date("event_time"))
    )

    query = (
        stream_df.writeStream
        .format("parquet")
        .option("path", f"s3a://{BUCKET}/processed/streaming_bronze/")
        .option("checkpointLocation", f"s3a://{BUCKET}/checkpoints/streaming_bronze/")
        .partitionBy("ingest_date")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
    print("Streaming micro-batch complete.")
    # Window-based scoring then runs over streaming_bronze exactly as in steps 3-4.